# Conformal Prediction Correction

We saw the 80% interval only gives about 74% coverage past day 1.
Here we fix that using conformalized quantile regression (CQR).

Idea: measure how wrong the intervals were on a calibration set,
then stretch the intervals by that amount so they hit the target coverage.

In [1]:
import numpy as np

In [3]:
forecasts = np.load("../data/forecasts.npy")
actuals = np.load("../data/actuals.npy")

forecasts.shape, actuals.shape

((352, 30, 11), (352, 30))

## Pick a horizon to correct

Let's fix the day 16 ahead forecast (index 15), since we saw that one
was overconfident (74.1% instead of 80%).

In [3]:
day = 15
quantile_levels = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]

day_forecasts = forecasts[:, day, :]
day_actuals = actuals[:, day]

lower = day_forecasts[:, 1]   # 10th percentile
upper = day_forecasts[:, 9]   # 90th percentile

day_forecasts.shape, day_actuals.shape

((352, 11), (352,))

## Split into calibration set and test set

Since this is time series data, we split by time, not randomly.
Earlier windows become the calibration set, later windows become the test set.

In [4]:
n = len(day_actuals)
split_point = int(n * 0.6)   # 60% calibration, 40% test

cal_lower, test_lower = lower[:split_point], lower[split_point:]
cal_upper, test_upper = upper[:split_point], upper[split_point:]
cal_actuals, test_actuals = day_actuals[:split_point], day_actuals[split_point:]

print("calibration size:", len(cal_actuals))
print("test size:", len(test_actuals))

calibration size: 211
test size: 141


## Compute conformity scores on the calibration set

For each calibration point, we check how far the actual value fell
outside the predicted interval.

If the actual was above the upper bound, the score is how far above.
If the actual was below the lower bound, the score is how far below.
If it was inside the interval, the score is negative (meaning there
was room to spare).

In [5]:
scores = np.maximum(cal_lower - cal_actuals, cal_actuals - cal_upper)
scores[:10]

array([ 266.58398438, -455.91242188, -229.65015625, -805.05054688,
       -490.68960938, -900.24515625, -361.27472656,  242.04605469,
       -908.75914062, -597.934375  ])

## Find the correction amount

We want 80% coverage, so we take the 80th percentile of these scores.
This tells us how much cushion to add so that 80% of calibration points
would have been covered.

In [6]:
target_coverage = 0.8
q_hat = np.quantile(scores, target_coverage)
q_hat

np.float64(117.59328124999956)

## Apply the correction to the test set

Stretch both the lower and upper bound by q_hat.

In [7]:
corrected_lower = test_lower - q_hat
corrected_upper = test_upper + q_hat

## Compare coverage: before vs after correction, on the test set

In [8]:
# before correction
inside_before = (test_actuals >= test_lower) & (test_actuals <= test_upper)
coverage_before = inside_before.mean()

# after correction
inside_after = (test_actuals >= corrected_lower) & (test_actuals <= corrected_upper)
coverage_after = inside_after.mean()

print("target coverage:", target_coverage)
print("coverage before correction:", round(coverage_before * 100, 1), "%")
print("coverage after correction:", round(coverage_after * 100, 1), "%")

target coverage: 0.8
coverage before correction: 75.2 %
coverage after correction: 78.0 %


## Check the cost: how much wider are the intervals now?

Conformal prediction guarantees coverage, but the price is wider intervals.
Let's see how much wider.

In [9]:
avg_width_before = (test_upper - test_lower).mean()
avg_width_after = (corrected_upper - corrected_lower).mean()

print("average interval width before:", round(avg_width_before, 2))
print("average interval width after:", round(avg_width_after, 2))
print("increase:", round((avg_width_after / avg_width_before - 1) * 100, 1), "%")

average interval width before: 5152.03
average interval width after: 5387.22
increase: 4.6 %


## Results and Interpretation

**Setup:** Chronos-2 was evaluated on Sensex daily closing prices (2010–2026) using
a rolling window forecast (512 days of history, 30 days ahead), giving 352 separate
forecasts to evaluate. The model outputs quantile forecasts (5th to 95th percentile),
not just a single point prediction.

**Finding 1: Coverage is well calibrated at very short horizons, but drops quickly.**

| Days ahead | Claimed coverage | Empirical coverage |
|---|---|---|
| 1  | 80% | 80.7% |
| 2  | 80% | 73.6% |
| 16 | 80% | 74.1% |
| 30 | 80% | 74.4% |

Chronos-2's stated 80% prediction interval is accurate one day ahead, but drops to
roughly 74% coverage from day 2 onward and stays flat around there rather than
degrading gradually. This means the model is overconfident at every horizon beyond
the next day — its intervals are narrower than they should be to hit the coverage
it claims.

**Finding 2: The miscalibration is directionally skewed, not just too narrow.**

PIT histograms (day 1 and beyond) show more mass in the upper range (0.7-0.95)
than the lower range, meaning the actual Sensex value lands above the model's
median forecast more often than a well-calibrated model would predict. This lines
up with Sensex's long-term upward trend over the sample period — the model
appears to slightly underestimate upside movement.

**Correction: split conformal prediction (CQR)**

Using the day-16-ahead forecasts, the data was split by time (first 60% as a
calibration set, last 40% as a test set). Conformity scores were computed as the
signed distance of each calibration actual outside its predicted interval, and the
80th percentile of these scores was used to widen the test set intervals.

| | Before correction | After correction |
|---|---|---|
| Coverage | 75.2% | 78.0% |
| Avg. interval width | 5152.03 | 5387.22 (+4.6%) |

Conformal prediction closed most of the coverage gap (75.2% -> 78.0%) at a small
cost in interval width (+4.6%). It did not fully close the gap to the 80% target,
which is a meaningful result in itself: standard conformal prediction assumes
calibration and test data are exchangeable (drawn from the same distribution).
The residual gap is consistent with a violation of this assumption -- Sensex's
statistical properties (volatility, trend) likely shifted between the earlier
calibration period and the later test period, since financial time series are
non-stationary by nature.

**Conclusion:** Chronos-2 shows good short-horizon calibration on Sensex but becomes
measurably overconfident beyond a 1-day horizon, with a directional bias toward
underestimating upside moves. Split conformal prediction partially corrects this at
low cost, but full correction likely requires a method designed for non-stationary
data, such as adaptive conformal prediction (Gibbs & Candes, 2021), which updates
the correction over time rather than fixing it once from a static calibration set.